---


  T96 Model Analysis Using the 9 Directional Derivative Formulas

  (8-panel visualization + extra panels)


  NOTE:

   - Output (values, figures, titles, etc.) is unchanged; only duplication and structure are cleaned up.

   - Formatted so cells can be split with # %% / # %% [markdown].


---

---

PNG saving utilities

---

In [ ]:
import os
import re
import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable

from geopack import (
    recalc,
    dip,
    smgsm_vectorized,
    t96_vectorized,
    field_line_directional_derivatives,
    verify_antisymmetry_relations,
    get_curvature_torsion_from_derivatives,
)

SAVE_PNG = True           # Save ON/OFF
PNG_DIR = "png_outputs"   # Output folder
PNG_DPI = 300             # Resolution
PNG_VERBOSE = False       # If True, print save path

_png_counter = 0


def _sanitize_filename(s, maxlen=180):
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    # Keep only safe characters for Windows, etc. (Japanese chars may be replaced with "_")
    s = re.sub(r"[^\w\-. ]+", "_", s)
    s = s.replace(" ", "_")
    return s[:maxlen]


def _build_short_name(content, plane, case_label, by_nT):
    """Build a clean filename like T96_Derivs_Meridional_GSM_Base_By0nT"""
    by_str = f"By{by_nT:g}nT".replace("-", "minus")
    return f"T96_{content}_{plane}_{case_label}_{by_str}"


def save_fig_png(fig, name, out_dir=PNG_DIR, dpi=PNG_DPI, short_name=None):
    global _png_counter
    if not SAVE_PNG:
        return None
    os.makedirs(out_dir, exist_ok=True)
    _png_counter += 1
    if short_name:
        base = _sanitize_filename(short_name)
        path = os.path.join(out_dir, f"{base}.png")
    else:
        base = _sanitize_filename(name)
        path = os.path.join(out_dir, f"{_png_counter:03d}_{base}.png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    if PNG_VERBOSE:
        print(f"Saved PNG: {path}")
    return path


def finish_figure(fig, name, show=True, close=True, short_name=None):
    save_fig_png(fig, name, short_name=short_name)
    if show:
        plt.show()
    if close:
        plt.close(fig)


---

User-tunable parameters

---

In [ ]:
delta_fd = 1e-5     # finite difference step (Re)
delta_min = 1e-5    # small-value clipping for derivative fields (display)

L0 = 0.5            # segment length scale (Re)
step = 2            # subsampling for overlay density

perp_thresh = 0.05  # |B_in_plane|/|B| < this -> dot instead of segment
dot_size = 4
dot_alpha = 0.85

SHOW_CHRISTOFFEL = False  # Show Christoffel symbol notation alongside derivative notation

# PNG naming: True -> clean short names (e.g. T96_Derivs_Meridional_GSM_Base_By0nT.png)
#             False -> sequential counter + sanitized title (e.g. 001_T96__The_8_..._Pdyn_....png)
USE_SHORT_NAMES = True

# Layout knobs
CBAR_PAD = 0.18
CBAR_SIZE = "4.5%"
TITLE_FONTSIZE = 16
TIGHT_HPAD = 1.2
TIGHT_WPAD = 0.9

UNIT_STR = r"$1/R_{\mathrm{E}}$"   # e.g. "1/Re" or r"1/R$_E$"


---

Keys / panels

---

In [ ]:
main_keys = [
    "dT_dT_n",
    "dT_dT_b",
    "dn_dT_b",
    "dT_dn_n",
    "dT_dn_b",
    "dn_dn_b",
    "dn_db_b",
    "dn_db_T",
    "db_db_T",
]
anti_keys = [
    "dn_dT_T",
    "db_dT_T",
    "db_dT_n",
    "dn_dn_T",
    "db_dn_T",
    "db_dn_n",
    "db_db_n",
    "dT_db_n",
    "dT_db_b",
]


def _gamma(sym):
    """Include Christoffel symbol if SHOW_CHRISTOFFEL is True."""
    return f" {sym}" if SHOW_CHRISTOFFEL else ""


plot_info_8 = [
    (
        "dT_dT_n",
        r"(∂T/∂T)·n" + _gamma(r"$=\Gamma^{\hat{2}}_{\hat{1}\hat{1}}$"),
        "Parallel Curvature",
        "plasma",
    ),
    (
        "db_dT_n",
        r"(∂b/∂T)·n" + _gamma(r"$=\Gamma^{\hat{2}}_{\hat{1}\hat{3}}$"),
        "Parallel Torsion",
        "RdBu_r",
    ),
    (
        "dT_dn_n",
        r"(∂T/∂n)·n" + _gamma(r"$=\Gamma^{\hat{2}}_{\hat{2}\hat{1}}$"),
        "Normal Curvature",
        "RdBu_r",
    ),
    (
        "dT_db_b",
        r"(∂T/∂b)·b" + _gamma(r"$=\Gamma^{\hat{3}}_{\hat{3}\hat{1}}$"),
        "Binormal Curvature",
        "RdBu_r",
    ),
    (
        "dn_db_b",
        r"(∂n/∂b)·b" + _gamma(r"$=\Gamma^{\hat{3}}_{\hat{3}\hat{2}}$"),
        "Binormal Curvature",
        "RdBu_r",
    ),
    (
        "db_dn_n",
        r"(∂b/∂n)·n" + _gamma(r"$=\Gamma^{\hat{2}}_{\hat{2}\hat{3}}$"),
        "Normal Curvature",
        "RdBu_r",
    ),
    (
        "dT_dn_b",
        r"(∂T/∂n)·b" + _gamma(r"$=\Gamma^{\hat{3}}_{\hat{2}\hat{1}}$"),
        "Normal Torsion",
        "RdBu_r",
    ),
    (
        "dT_db_n",
        r"(∂T/∂b)·n" + _gamma(r"$=\Gamma^{\hat{2}}_{\hat{3}\hat{1}}$"),
        "Binormal Torsion",
        "RdBu_r",
    ),
]


---

Utilities

---

In [ ]:
def get_max_error(error_val):
    if hasattr(error_val, "shape"):
        return np.nanmax(np.abs(error_val))
    return abs(error_val)


def robust_vmin_vmax(data, param_name):
    """Return (vmin, vmax) suitable for stable plotting."""
    data = np.asarray(data)

    if param_name == "dT_dT_n":  # curvature: non-negative
        vmax = np.nanpercentile(data, 95)
        if (not np.isfinite(vmax)) or (vmax <= 0):
            vmax = np.nanmax(data)
        if (not np.isfinite(vmax)) or (vmax <= 0):
            vmax = 1e-12
        vmin = 0.0
        return vmin, vmax

    vmax = np.nanpercentile(np.abs(data), 95)
    if (not np.isfinite(vmax)) or (vmax <= 0):
        vmax = np.nanmax(np.abs(data))
    if (not np.isfinite(vmax)) or (vmax <= 0):
        vmax = 1e-12
    vmin = -vmax
    return vmin, vmax


def plot_scalar_on_grid(ax, X, Y, data, cmap, param_name, force_levels=None):
    """
    Safe plotting helper:
      - uses contourf when range exists,
      - falls back to imshow when data is (almost) constant,
      - guarantees increasing contour levels.
    Returns mappable for colorbar.
    """
    data = np.asarray(data)

    if force_levels is not None:
        levels = force_levels
        return ax.contourf(X, Y, data, levels=levels, cmap=cmap, extend="both")

    vmin, vmax = robust_vmin_vmax(data, param_name)

    dmin = np.nanmin(data)
    dmax = np.nanmax(data)
    if (not np.isfinite(dmin)) or (not np.isfinite(dmax)) or (abs(dmax - dmin) < 1e-14):
        extent = (np.nanmin(X), np.nanmax(X), np.nanmin(Y), np.nanmax(Y))
        im = ax.imshow(
            data,
            origin="lower",
            extent=extent,
            aspect="equal",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
        )
        return im

    if vmax <= vmin:
        eps = 1e-12 if vmin == 0 else abs(vmin) * 1e-6 + 1e-12
        vmin -= eps
        vmax += eps

    levels = np.linspace(vmin, vmax, 20)
    data_for_plot = np.clip(data, vmin, vmax)
    return ax.contourf(X, Y, data_for_plot, levels=levels, cmap=cmap, vmin=vmin, vmax=vmax)


def add_colorbar(fig, ax, im, label, unit=None):
    """
    unit:
      - None -> uses UNIT_STR (for Gamma, etc.)
      - ""   -> no unit display (for ratio, etc.)
      - "A/m$^2$" -> [A/m$^2$] (for FAC current density, etc.)
    """
    if unit is None:
        unit = UNIT_STR
    else:
        # Absorb all existing unit="1/Re" into UNIT_STR for consistency
        if isinstance(unit, str) and unit.strip() == "1/Re":
            unit = UNIT_STR

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size=CBAR_SIZE, pad=CBAR_PAD)
    cbar = fig.colorbar(im, cax=cax)

    if unit != "":
        cbar.set_label(f"{label} $\!$ [{unit}]", fontsize=10)
    else:
        cbar.set_label(f"{label}", fontsize=10)

    try:
        cbar.ax.yaxis.get_offset_text().set_fontsize(9)
    except Exception:
        pass

    return cbar


def set_panel_title(ax, formula, title):
    ax.set_title(f"{formula}\n{title}", fontsize=TITLE_FONTSIZE, loc="left", pad=6)
    ax.title.set_multialignment("left")


def add_micro_arrows(
    ax,
    segments,
    color="k",
    alpha=0.8,
    width=0.0022,      # thin
    headwidth=2.6,     # small arrowhead triangle
    headlength=3.2,
    headaxislength=3.4,
    zorder=5,
):
    """
    segments: shape (N,2,2), a collection of [ [x0,y0], [x1,y1] ]
    Uses the center of each segment as the origin with pivot='middle',
    so the arrow and segment have the same total length (length invariant).
    """
    if segments is None:
        return None

    seg = np.asarray(segments)
    x0, y0 = seg[:, 0, 0], seg[:, 0, 1]
    x1, y1 = seg[:, 1, 0], seg[:, 1, 1]

    xc = 0.5 * (x0 + x1)
    yc = 0.5 * (y0 + y1)
    u = x1 - x0
    v = y1 - y0

    return ax.quiver(
        xc,
        yc,
        u,
        v,
        angles="xy",
        scale_units="xy",
        scale=1,          # use length as-is in data coordinates
        pivot="middle",   # center-aligned -> same length as segment
        color=color,
        alpha=alpha,
        width=width,
        headwidth=headwidth,
        headlength=headlength,
        headaxislength=headaxislength,
        zorder=zorder,
    )


def _clip_small(arr, thr):
    arr = np.asarray(arr)
    arr[np.abs(arr) < thr] = 0
    return arr


def reshape_and_clip_derivatives(derivatives, shape, keys):
    params = {}
    for k in keys:
        a = derivatives[k].reshape(shape)
        params[k] = _clip_small(a, delta_min)
    return params


def print_antisymmetry(errors, header):
    print(header)
    for name, err in errors.items():
        print(f"{name:20} = {get_max_error(err):.2e}")



---

T96 setup

---

In [ ]:
# From date and time
t1 = datetime.datetime(2001, 3, 22, 0, 0, 0)
t0 = datetime.datetime(1970, 1, 1)
ut = (t1 - t0).total_seconds()

ut = 5000.0 + 90.0 * 24.0 * 60.0 * 60.0
ps = recalc(ut)
print(f"Dipole tilt angle (ps): {np.degrees(ps):.2f} degrees")


def make_parmod_from_sw(density_cm3, v_kms, dst_nT, by_nT, bz_nT):
    # same recipe as original
    density_m3 = density_cm3 * 1e6
    velocity_ms = v_kms * 1e3
    Pdyn = 1.6726219e-27 * density_m3 * velocity_ms**2 / 1e-9  # nPa
    parmod = [Pdyn, dst_nT, by_nT, bz_nT, 0, 0, 0, 0, 0, 0]
    return parmod, Pdyn



---

Field wrapper (T96 external + internal dipole)

---

In [ ]:
def t96_total_field(parmod_in, ps_in, x, y, z):
    bx_ex, by_ex, bz_ex = t96_vectorized(parmod_in, ps_in, x, y, z)
    bx_in, by_in, bz_in = dip(x, y, z)
    return bx_ex + bx_in, by_ex + by_in, bz_ex + bz_in


def t96_field_wrapper(parmod_in, ps_in, x, y, z):
    return t96_total_field(parmod_in, ps_in, x, y, z)



---

Overlay builders (same behavior)

---

In [ ]:
def build_overlay_segments_and_dots_meridional_field(X, Z, Y_plane, field_func, parmod_in, ps_in):
    """
    GSM X-Z plane (Y fixed):
      - segments aligned with in-plane component (Bx, Bz)
      - dots when in-plane component is tiny vs |B|
    """
    Bx, By, Bz = field_func(parmod_in, ps_in, X, Y_plane, Z)

    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
    Bmag[Bmag == 0] = np.nan

    Bin = np.sqrt(Bx**2 + Bz**2)
    ratio = Bin / Bmag

    Bin_safe = Bin.copy()
    Bin_safe[Bin_safe == 0] = np.nan

    dx = L0 * Bx / Bin_safe
    dz = L0 * Bz / Bin_safe

    Xs = X[::step, ::step]
    Zs = Z[::step, ::step]
    dxs = dx[::step, ::step]
    dzs = dz[::step, ::step]
    rs = ratio[::step, ::step]

    finite = np.isfinite(dxs) & np.isfinite(dzs) & np.isfinite(rs)
    mask_line = finite & (rs >= perp_thresh)
    mask_dot = finite & (rs < perp_thresh)

    if np.any(mask_line):
        x0 = Xs[mask_line] - dxs[mask_line] / 2.0
        z0 = Zs[mask_line] - dzs[mask_line] / 2.0
        x1 = Xs[mask_line] + dxs[mask_line] / 2.0
        z1 = Zs[mask_line] + dzs[mask_line] / 2.0
        segments = np.stack(
            [np.stack([x0, z0], axis=-1), np.stack([x1, z1], axis=-1)], axis=1
        )
    else:
        segments = None

    dot_x = Xs[mask_dot]
    dot_z = Zs[mask_dot]
    return segments, dot_x, dot_z


def build_overlay_segments_and_dots_equator_sm_field(
    X_sm, Y_sm, X_gsm, Y_gsm, Z_gsm, field_func, parmod_in, ps_in
):
    """
    SM equator plot (SM X-Y plane, Z_SM=0):
      - compute B at GSM points on that surface
      - rotate vector GSM→SM
      - segments from in-plane (Bx_sm, By_sm)
      - dots when |B_in_plane|/|B| is tiny
    """
    Bx_g, By_g, Bz_g = field_func(parmod_in, ps_in, X_gsm, Y_gsm, Z_gsm)
    Bx_s, By_s, Bz_s = smgsm_vectorized(Bx_g, By_g, Bz_g, j=-1)

    Bmag = np.sqrt(Bx_s**2 + By_s**2 + Bz_s**2)
    Bmag[Bmag == 0] = np.nan

    Bxy = np.sqrt(Bx_s**2 + By_s**2)
    ratio = Bxy / Bmag

    Bxy_safe = Bxy.copy()
    Bxy_safe[Bxy_safe == 0] = np.nan

    dx = np.full_like(Bx_s, np.nan, dtype=float)
    dy = np.full_like(By_s, np.nan, dtype=float)

    mask_line_full = np.isfinite(ratio) & (ratio >= perp_thresh) & np.isfinite(Bxy_safe)
    dx[mask_line_full] = L0 * Bx_s[mask_line_full] / Bxy_safe[mask_line_full]
    dy[mask_line_full] = L0 * By_s[mask_line_full] / Bxy_safe[mask_line_full]

    Xs = X_sm[::step, ::step]
    Ys = Y_sm[::step, ::step]
    dxs = dx[::step, ::step]
    dys = dy[::step, ::step]
    rs = ratio[::step, ::step]

    finite = np.isfinite(rs)
    mask_line = finite & (rs >= perp_thresh) & np.isfinite(dxs) & np.isfinite(dys)
    mask_dot = finite & (rs < perp_thresh)

    if np.any(mask_line):
        x0 = Xs[mask_line] - dxs[mask_line] / 2.0
        y0 = Ys[mask_line] - dys[mask_line] / 2.0
        x1 = Xs[mask_line] + dxs[mask_line] / 2.0
        y1 = Ys[mask_line] + dys[mask_line] / 2.0
        segments = np.stack(
            [np.stack([x0, y0], axis=-1), np.stack([x1, y1], axis=-1)], axis=1
        )
    else:
        segments = None

    dot_x = Xs[mask_dot]
    dot_y = Ys[mask_dot]
    return segments, dot_x, dot_y



---

Grids

---

In [ ]:
# Meridional plane (GSM): X-Z at Y=0
x_mer = np.linspace(-8, 8, 60)
z_mer = np.linspace(-6, 6, 50)
X_mer, Z_mer = np.meshgrid(x_mer, z_mer)
Y_mer = np.zeros_like(X_mer)

# Magnetic equator (SM): Z_SM=0 (plot plane is SM X-Y)
x_sm = np.linspace(-8, 8, 60)
y_sm = np.linspace(-10, 10, 60)
X_sm, Y_sm = np.meshgrid(x_sm, y_sm)
Z_sm = np.zeros_like(X_sm)

# SM -> GSM (compute plane)
X_eq_gsm, Y_eq_gsm, Z_eq_gsm = smgsm_vectorized(X_sm, Y_sm, Z_sm, j=+1)

# Flatten for vectorized derivative computation
x_mer_flat = X_mer.ravel()
y_mer_flat = Y_mer.ravel()
z_mer_flat = Z_mer.ravel()

x_eq_flat = X_eq_gsm.ravel()
y_eq_flat = Y_eq_gsm.ravel()
z_eq_flat = Z_eq_gsm.ravel()

print(f"\nMeridional GSM grid: {X_mer.shape}")
print(f"Equator SM grid (Z_SM=0): {X_sm.shape}  -> computed in GSM")



---

Common compute / plot helpers

---

In [ ]:
def compute_derivatives_flat(parmod_in, ps_in, x_flat, y_flat, z_flat):
    return field_line_directional_derivatives(
        t96_field_wrapper,
        parmod_in,
        ps_in,
        x_flat,
        y_flat,
        z_flat,
        delta=delta_fd,
    )


def plot_8_panels(
    X,
    Y,
    params,
    segments,
    dot_x,
    dot_y,
    xlabel,
    ylabel,
    xlim,
    ylim,
    suptitle,
    sw_title_line,
    figsize,
    add_earth=True,
    earth_radius=1.0,  # existing output unchanged with default 1.0
    short_name=None,
):
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(4, 2, figure=fig, hspace=0.35, wspace=0.25)

    for idx, (param_name, formula, title, cmap) in enumerate(plot_info_8):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        data = params[param_name]

        if param_name == "dn_db_b" and np.nanmax(np.abs(data)) <= 1.2:
            im = plot_scalar_on_grid(
                ax,
                X,
                Y,
                data,
                cmap,
                "dn_db_b",
                force_levels=np.linspace(-1, 1, 21),
            )
        else:
            im = plot_scalar_on_grid(ax, X, Y, data, cmap, param_name)

        if segments is not None:
            add_micro_arrows(ax, segments, alpha=0.8)
        if dot_x.size > 0:
            ax.scatter(dot_x, dot_y, s=dot_size, marker="o", c="k", alpha=dot_alpha, linewidths=0)

        if add_earth:
            ax.add_patch(plt.Circle((0, 0), earth_radius, color="white", zorder=10))  # <-- variable radius

        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        set_panel_title(ax, formula, title)
        ax.set_aspect("equal")
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)

        add_colorbar(fig, ax, im, formula)

        ax.text(
            0.02,
            0.98,
            f"[{np.nanmin(data):.2e}, {np.nanmax(data):.2e}]",
            transform=ax.transAxes,
            fontsize=9,
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    fig.tight_layout(h_pad=TIGHT_HPAD, w_pad=TIGHT_WPAD, pad=0.3)
    fig.subplots_adjust(top=0.93)
    fig.suptitle(f"{suptitle}\n{sw_title_line}", fontsize=16, y=0.985)

    finish_figure(fig, f"{suptitle}__{sw_title_line}", show=True, close=True, short_name=short_name)


---

ADD: SM z = const plane (parallel to magnetic equator), computed in GSM

---

In [ ]:
def _earth_radius_for_sm_z(z_sm0):
    # Earth sphere radius=1 Re, plane z=z_sm0 -> cross-section radius sqrt(1-z^2)
    val = 1.0 - float(z_sm0) ** 2
    return np.sqrt(val) if val > 0 else 0.0


def compute_derivatives_on_sm_z_plane(parmod_in, z_sm0):
    """
    SM plane: Z_SM=z_sm0 (plot plane is SM X-Y).
    Compute in GSM by rotating coordinates SM->GSM.
    """
    Z_sm_plane = np.full_like(X_sm, z_sm0, dtype=float)
    X_gsm, Y_gsm, Z_gsm = smgsm_vectorized(X_sm, Y_sm, Z_sm_plane, j=+1)

    derivatives = compute_derivatives_flat(
        parmod_in,
        ps,
        X_gsm.ravel(),
        Y_gsm.ravel(),
        Z_gsm.ravel(),
    )
    return derivatives, X_gsm, Y_gsm, Z_gsm


def plot_8_panels_sm_z_plane(parmod_in, z_sm0, sw_title_line, suptitle, figsize=(16, 22), short_name=None):
    derivatives, X_gsm, Y_gsm, Z_gsm = compute_derivatives_on_sm_z_plane(parmod_in, z_sm0)

    params = reshape_and_clip_derivatives(derivatives, X_sm.shape, keys=(main_keys + anti_keys))

    segments, dot_x, dot_y = build_overlay_segments_and_dots_equator_sm_field(
        X_sm,
        Y_sm,
        X_gsm,
        Y_gsm,
        Z_gsm,
        t96_total_field,
        parmod_in,
        ps,
    )

    r_earth = _earth_radius_for_sm_z(z_sm0)

    plot_8_panels(
        X_sm,
        Y_sm,
        params,
        segments,
        dot_x,
        dot_y,
        xlabel="X_SM (Re)",
        ylabel="Y_SM (Re)",
        xlim=(x_sm.min(), x_sm.max()),
        ylim=(y_sm.min(), y_sm.max()),
        suptitle=suptitle,
        sw_title_line=sw_title_line,
        figsize=figsize,
        add_earth=True,
        earth_radius=r_earth,  # √(1 - z_sm0^2)
        short_name=short_name,
    )


def compute_fac_proxy_and_ratio(Bmag_nT, dn_db_T, db_dn_T):
    mu0 = 1.25663706127e-6
    FAC = Bmag_nT * (dn_db_T - db_dn_T) / mu0 * (1e-9) / (6371 * 1e3)  # A/m^2 (1 Re = 6371 km = 6.371e6 m)
    den = np.abs(dn_db_T) + np.abs(db_dn_T)
    ratio = np.abs(dn_db_T) / np.where(den == 0, np.nan, den)
    return FAC, ratio


def plot_fac_4panels_sm_z_plane(parmod_in, z_sm0, sw_title_line, suptitle_prefix, short_name=None):
    """
    FAC-related 4 panels on SM X-Y plane at Z_SM=z_sm0 (computed in GSM)
    (display style follows the existing equator 4-panel layout)
    """
    derivatives, X_gsm, Y_gsm, Z_gsm = compute_derivatives_on_sm_z_plane(parmod_in, z_sm0)

    dn_db_T = _clip_small(derivatives["dn_db_T"].reshape(X_sm.shape), delta_min)
    db_dn_T = _clip_small(derivatives["db_dn_T"].reshape(X_sm.shape), delta_min)

    bx, by, bz = t96_total_field(parmod_in, ps, X_gsm, Y_gsm, Z_gsm)
    Bmag_nT = np.sqrt(bx**2 + by**2 + bz**2)

    FAC, ratio = compute_fac_proxy_and_ratio(Bmag_nT, dn_db_T, db_dn_T)

    segments, dot_x, dot_y = build_overlay_segments_and_dots_equator_sm_field(
        X_sm,
        Y_sm,
        X_gsm,
        Y_gsm,
        Z_gsm,
        t96_total_field,
        parmod_in,
        ps,
    )

    r_earth = _earth_radius_for_sm_z(z_sm0)

    fig = plt.figure(figsize=(18, 17))
    gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.25)
    axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]

    panels = [
        dict(
            data=dn_db_T,
            cmap="RdBu_r",
            pname="residual",
            formula="",
            title=r"(∂n/∂b)·T" + _gamma(r"$=\Gamma^{\hat{1}}_{\hat{3}\hat{2}}$"),
            cbar_label=r"(∂n/∂b)·T",
            unit="1/Re",
            force_levels=None,
        ),
        dict(
            data=-db_dn_T,
            cmap="RdBu_r",
            pname="residual",
            formula="",
            title=r"-(∂b/∂n)·T" + _gamma(r"$=-\Gamma^{\hat{1}}_{\hat{2}\hat{3}}$"),
            cbar_label=r"(∂b/∂n)·T",
            unit="1/Re",
            force_levels=None,
        ),
        dict(
            data=FAC,
            cmap="RdBu_r",
            pname="residual",
            formula="",
            title=(
                r"Field-Aligned Current  $\frac{B}{\mu_0}$((∂n/∂b)·T − (∂b/∂n)·T)"
            ),
            cbar_label="FAC",
            unit="A/m$^2$",
            force_levels=None,
        ),
        dict(
            data=ratio,
            cmap="viridis",
            pname="ratio",
            formula=r"$\frac{|dn\_db\_T|}{|dn\_db\_T|+|db\_dn\_T|}$",
            title="Contribution ratio (0..1)",
            cbar_label="ratio",
            unit="",
            force_levels=np.linspace(0, 1, 21),
        ),
    ]

    for ax, P in zip(axes, panels):
        data = P["data"]
        if P["force_levels"] is not None:
            im = plot_scalar_on_grid(
                ax,
                X_sm,
                Y_sm,
                data,
                P["cmap"],
                P["pname"],
                force_levels=P["force_levels"],
            )
        else:
            im = plot_scalar_on_grid(ax, X_sm, Y_sm, data, P["cmap"], P["pname"])

        if segments is not None:
            add_micro_arrows(ax, segments, alpha=0.8)
        if dot_x.size > 0:
            ax.scatter(dot_x, dot_y, s=dot_size, marker="o", c="k", alpha=dot_alpha, linewidths=0)

        ax.add_patch(plt.Circle((0, 0), r_earth, color="white", zorder=10))  # √(1 - z_sm0^2)
        ax.set_xlabel("X_SM (Re)")
        ax.set_ylabel("Y_SM (Re)")
        set_panel_title(ax, P["formula"], P["title"])
        ax.set_aspect("equal")
        ax.set_xlim(np.nanmin(X_sm), np.nanmax(X_sm))
        ax.set_ylim(np.nanmin(Y_sm), np.nanmax(Y_sm))

        add_colorbar(fig, ax, im, P["cbar_label"], unit=P["unit"])

        ax.text(
            0.02,
            0.98,
            f"[{np.nanmin(data):.2e}, {np.nanmax(data):.2e}]",
            transform=ax.transAxes,
            fontsize=9,
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    fig.tight_layout(h_pad=TIGHT_HPAD, w_pad=TIGHT_WPAD, pad=0.3)
    fig.subplots_adjust(top=0.92)
    fig.suptitle(
        f"{suptitle_prefix}\nSM plane at Z_SM={z_sm0:g} (computed in GSM)\n{sw_title_line}",
        fontsize=16,
        y=0.985,
    )

    finish_figure(fig, f"{suptitle_prefix}__SM_z{z_sm0:g}__{sw_title_line}", show=True, close=True, short_name=short_name)


---

ADD: GSM X-Z plane at fixed Y_GSM utilities

---

In [ ]:
def compute_derivatives_on_xz_at_y(parmod_in, y0):
    X = X_mer
    Z = Z_mer
    Y = np.full_like(X, y0, dtype=float)

    derivatives = field_line_directional_derivatives(
        t96_field_wrapper,
        parmod_in,
        ps,
        X.ravel(),
        Y.ravel(),
        Z.ravel(),
        delta=delta_fd,
    )
    return derivatives, X, Y, Z


def plot_mainkeys_xz_at_y(
    parmod_in,
    y0,
    sw_title_line,
    suptitle_prefix="T96: The 9 Directional Derivative Formulas",
    short_name=None,
):
    """
    (preserving original code behavior)
    GSM X-Z plane at Y_GSM=y0: 2-column (4x2) 8 panels
    """
    X = X_mer
    Z = Z_mer
    Y = np.full_like(X, y0, dtype=float)

    derivatives = field_line_directional_derivatives(
        t96_field_wrapper,
        parmod_in,
        ps,
        X.ravel(),
        Y.ravel(),
        Z.ravel(),
        delta=delta_fd,
    )

    params = reshape_and_clip_derivatives(derivatives, X.shape, keys=(main_keys + anti_keys))

    segments, dot_x, dot_z = build_overlay_segments_and_dots_meridional_field(
        X,
        Z,
        Y,
        t96_total_field,
        parmod_in,
        ps,
    )

    fig = plt.figure(figsize=(18, 24))
    gs = GridSpec(4, 2, figure=fig, hspace=0.35, wspace=0.25)

    for idx, (key, formula, title, cmap) in enumerate(plot_info_8):
        ax = fig.add_subplot(gs[idx // 2, idx % 2])
        data = params[key]

        if key == "dn_db_b" and np.nanmax(np.abs(data)) <= 1.2:
            im = plot_scalar_on_grid(
                ax,
                X,
                Z,
                data,
                cmap,
                "dn_db_b",
                force_levels=np.linspace(-1, 1, 21),
            )
        else:
            im = plot_scalar_on_grid(ax, X, Z, data, cmap, key)

        if segments is not None:
            add_micro_arrows(ax, segments, alpha=0.8)
        if dot_x.size > 0:
            ax.scatter(dot_x, dot_z, s=dot_size, marker="o", c="k", alpha=dot_alpha, linewidths=0)

        ax.set_xlabel("X_GSM (Re)")
        ax.set_ylabel("Z_GSM (Re)")
        set_panel_title(ax, formula, f"{title}  (Y_GSM={y0:g})")
        ax.set_aspect("equal")
        ax.set_xlim(x_mer.min(), x_mer.max())
        ax.set_ylim(z_mer.min(), z_mer.max())

        add_colorbar(fig, ax, im, formula, unit="1/Re")

        ax.text(
            0.02,
            0.98,
            f"[{np.nanmin(data):.2e}, {np.nanmax(data):.2e}]",
            transform=ax.transAxes,
            fontsize=9,
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    fig.tight_layout(h_pad=TIGHT_HPAD, w_pad=TIGHT_WPAD, pad=0.3)
    fig.subplots_adjust(top=0.93)
    fig.suptitle(
        f"{suptitle_prefix}\nGSM X–Z plane at Y_GSM={y0:g}\n{sw_title_line}",
        fontsize=16,
        y=0.985,
    )

    finish_figure(fig, f"{suptitle_prefix}__GSM_XZ_Y{y0:g}__{sw_title_line}", show=True, close=True, short_name=short_name)


def plot_fac_4panels_xz_at_y(parmod_in, y0, suptitle_prefix, sw_title_line, short_name=None):
    """
    Plot FAC-related 4 panels on GSM X-Z plane at Y=y0 (unchanged from original code)
    """
    derivatives, X, Y, Z = compute_derivatives_on_xz_at_y(parmod_in, y0)

    dn_db_T_xz = _clip_small(derivatives["dn_db_T"].reshape(X.shape), delta_min)
    db_dn_T_xz = _clip_small(derivatives["db_dn_T"].reshape(X.shape), delta_min)

    bx, by, bz = t96_total_field(parmod_in, ps, X, Y, Z)
    Bmag_nT = np.sqrt(bx**2 + by**2 + bz**2)

    FAC_xz, ratio_xz = compute_fac_proxy_and_ratio(Bmag_nT, dn_db_T_xz, db_dn_T_xz)

    segments, dot_x, dot_z = build_overlay_segments_and_dots_meridional_field(
        X,
        Z,
        Y,
        t96_total_field,
        parmod_in,
        ps,
    )

    fig = plt.figure(figsize=(20, 15))
    gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.25)
    axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]

    panels = [
        dict(
            data=dn_db_T_xz,
            cmap="RdBu_r",
            pname="residual",
            formula="",
            title=r"(∂n/∂b)·T" + _gamma(r"$=\Gamma^{\hat{1}}_{\hat{3}\hat{2}}$"),
            cbar_label=r"(∂n/∂b)·T",
            unit="1/Re",
            force_levels=None,
        ),
        dict(
            data=-db_dn_T_xz,
            cmap="RdBu_r",
            pname="residual",
            formula="",
            title=r"-(∂b/∂n)·T" + _gamma(r"$=-\Gamma^{\hat{1}}_{\hat{2}\hat{3}}$"),
            cbar_label=r"(∂b/∂n)·T",
            unit="1/Re",
            force_levels=None,
        ),
        dict(
            data=FAC_xz,
            cmap="RdBu_r",
            pname="residual",
            formula="",
            title=(
                r"Field-Aligned Current  $\frac{B}{\mu_0}$((∂n/∂b)·T − (∂b/∂n)·T)"
            ),
            cbar_label="FAC",
            unit="A/m$^2$",
            force_levels=None,
        ),
        dict(
            data=ratio_xz,
            cmap="viridis",
            pname="ratio",
            formula=r"$\frac{|dn\_db\_T|}{|dn\_db\_T|+|db\_dn\_T|}$",
            title="Contribution ratio (0..1)",
            cbar_label="ratio",
            unit="",
            force_levels=np.linspace(0, 1, 21),
        ),
    ]

    for ax, P in zip(axes, panels):
        data = P["data"]
        if P["force_levels"] is not None:
            im = plot_scalar_on_grid(
                ax,
                X,
                Z,
                data,
                P["cmap"],
                P["pname"],
                force_levels=P["force_levels"],
            )
        else:
            im = plot_scalar_on_grid(ax, X, Z, data, P["cmap"], P["pname"])

        if segments is not None:
            add_micro_arrows(ax, segments, alpha=0.8)
        if dot_x.size > 0:
            ax.scatter(dot_x, dot_z, s=dot_size, marker="o", c="k", alpha=dot_alpha, linewidths=0)

        ax.set_xlabel("X_GSM (Re)")
        ax.set_ylabel("Z_GSM (Re)")
        set_panel_title(ax, P["formula"], f"{P['title']}")
        ax.set_aspect("equal")

        add_colorbar(fig, ax, im, P["cbar_label"], unit=P["unit"])

        ax.text(
            0.02,
            0.98,
            f"[{np.nanmin(data):.2e}, {np.nanmax(data):.2e}]",
            transform=ax.transAxes,
            fontsize=9,
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    fig.tight_layout(h_pad=TIGHT_HPAD, w_pad=TIGHT_WPAD, pad=0.3)
    fig.subplots_adjust(top=0.92)
    fig.suptitle(
        f"{suptitle_prefix}\nGSM X–Z plane at Y_GSM={y0:g}\n{sw_title_line}",
        fontsize=16,
        y=0.985,
    )

    finish_figure(fig, f"{suptitle_prefix}__GSM_XZ_Y{y0:g}__{sw_title_line}", show=True, close=True, short_name=short_name)


---

Wrapper to generate all figures for both BY=0nT and BY=5nT (added)

---

In [ ]:
BY_VALUES = [0.0, 5.0]


def run_all_figures_for_case(parmod_in, sw_title_line, case_mode="baseline"):
    """
    case_mode:
      - "baseline" : Original baseline SW conditions (display and print follow original flow)
      - "new"      : NEW SW conditions (display and print follow original flow)
    """
    case_label = "Base" if case_mode == "baseline" else "New"
    by_nT = parmod_in[2]

    def sn(content, plane):
        if not USE_SHORT_NAMES:
            return None
        return _build_short_name(content, plane, case_label, by_nT)

    # -------------------------
    # 1) MERIDIONAL PLANE (GSM Y=0)
    # -------------------------
    if case_mode == "baseline":
        print("\nCalculating directional derivatives (T96, Meridional plane GSM)...")
    else:
        print("\n[NEW SW] Calculating directional derivatives (T96, Meridional plane GSM)...")

    derivatives_mer = compute_derivatives_flat(parmod_in, ps, x_mer_flat, y_mer_flat, z_mer_flat)

    curvature_mer, torsion_mer = get_curvature_torsion_from_derivatives(derivatives_mer)
    if case_mode == "baseline":
        print(
            f"Curvature range (mer): {np.nanmin(curvature_mer):.3f} "
            f"to {np.nanmax(curvature_mer):.3f} 1/Re"
        )
        print(
            f"Torsion   range (mer): {np.nanmin(torsion_mer):.3e} "
            f"to {np.nanmax(torsion_mer):.3e} 1/Re"
        )
    else:
        print(
            f"[NEW SW] Curvature range (mer): {np.nanmin(curvature_mer):.3f} "
            f"to {np.nanmax(curvature_mer):.3f} 1/Re"
        )
        print(
            f"[NEW SW] Torsion   range (mer): {np.nanmin(torsion_mer):.3e} "
            f"to {np.nanmax(torsion_mer):.3e} 1/Re"
        )

    params_mer = reshape_and_clip_derivatives(derivatives_mer, X_mer.shape, keys=(main_keys + anti_keys))

    errors_mer = verify_antisymmetry_relations(derivatives_mer)
    if case_mode == "baseline":
        print_antisymmetry(errors_mer, "\nAntisymmetry verification (Meridional; max errors):")
    else:
        print_antisymmetry(errors_mer, "\n[NEW SW] Antisymmetry verification (Meridional; max errors):")

    segments_mer, dot_x_mer, dot_z_mer = build_overlay_segments_and_dots_meridional_field(
        X_mer,
        Z_mer,
        Y_mer,
        t96_total_field,
        parmod_in,
        ps,
    )

    plot_8_panels(
        X_mer,
        Z_mer,
        params_mer,
        segments_mer,
        dot_x_mer,
        dot_z_mer,
        xlabel="X_GSM (Re)",
        ylabel="Z_GSM (Re)",
        xlim=(x_mer.min(), x_mer.max()),
        ylim=(z_mer.min(), z_mer.max()),
        suptitle="T96: The 8 Directional Derivative Formulas (Meridional Plane, GSM)",
        sw_title_line=sw_title_line,
        figsize=(16, 22) if case_mode == "baseline" else (18, 22),
        add_earth=True,
        short_name=sn("Derivs", "Meridional_GSM"),
    )

    # -------------------------
    # ADD: 8 distributions on GSM X-Z at Y_GSM = -3
    # -------------------------
    if case_mode == "baseline":
        plot_mainkeys_xz_at_y(
            parmod_in, y0=-3.0, sw_title_line=sw_title_line,
            short_name=sn("Derivs", "Yminus3_GSM"),
        )
    else:
        plot_mainkeys_xz_at_y(
            parmod_in,
            y0=-3.0,
            sw_title_line=sw_title_line,
            suptitle_prefix="T96: The 8 Directional Derivative Formulas",
            short_name=sn("Derivs", "Yminus3_GSM"),
        )

    # -------------------------
    # 2) MAGNETIC EQUATOR (SM z=0 plane, computed in GSM)
    # -------------------------
    if case_mode == "baseline":
        print(
            "\nCalculating directional derivatives "
            "(T96, Magnetic equator: SM z=0 grid -> compute in GSM)..."
        )
    else:
        print("Recomputing directional derivatives on magnetic equator (SM z=0 grid -> GSM compute) ...")

    derivatives_eq = compute_derivatives_flat(parmod_in, ps, x_eq_flat, y_eq_flat, z_eq_flat)

    if case_mode == "baseline":
        curvature_eq, torsion_eq = get_curvature_torsion_from_derivatives(derivatives_eq)
        print(
            f"Curvature range (eq): {np.nanmin(curvature_eq):.3f} "
            f"to {np.nanmax(curvature_eq):.3f} 1/Re"
        )
        print(
            f"Torsion   range (eq): {np.nanmin(torsion_eq):.3e} "
            f"to {np.nanmax(torsion_eq):.3e} 1/Re"
        )

    params_eq = reshape_and_clip_derivatives(derivatives_eq, X_sm.shape, keys=(main_keys + anti_keys))

    errors_eq = verify_antisymmetry_relations(derivatives_eq)
    if case_mode == "baseline":
        print_antisymmetry(errors_eq, "\nAntisymmetry verification (Magnetic equator; max errors):")
    else:
        print_antisymmetry(errors_eq, "\n[NEW SW] Antisymmetry verification (Magnetic equator; max errors):")

    segments_eq, dot_x_eq, dot_y_eq = build_overlay_segments_and_dots_equator_sm_field(
        X_sm,
        Y_sm,
        X_eq_gsm,
        Y_eq_gsm,
        Z_eq_gsm,
        t96_total_field,
        parmod_in,
        ps,
    )

    plot_8_panels(
        X_sm,
        Y_sm,
        params_eq,
        segments_eq,
        dot_x_eq,
        dot_y_eq,
        xlabel="X_SM (Re)",
        ylabel="Y_SM (Re)",
        xlim=(x_sm.min(), x_sm.max()),
        ylim=(y_sm.min(), y_sm.max()),
        suptitle="T96: The 8 Directional Derivative Formulas (Magnetic Equator: SM z=0, computed in GSM)",
        sw_title_line=sw_title_line,
        figsize=(16, 24),
        add_earth=True,
        short_name=sn("Derivs", "Equator_SM"),
    )

    # -------------------------
    # ADD: SM z=0.5 plane
    # -------------------------
    plot_8_panels_sm_z_plane(
        parmod_in,
        z_sm0=0.5,
        sw_title_line=sw_title_line,
        suptitle="T96: The 8 Directional Derivative Formulas (SM z=0.5, computed in GSM)",
        figsize=(16, 24),
        short_name=sn("Derivs", "SMz05_SM"),
    )

    # -------------------------
    # Extra 4 panels (SM z=0): dn_db_T, db_dn_T, FAC proxy, ratio
    # -------------------------
    if case_mode == "baseline":
        dn_db_T = params_eq["dn_db_T"]
        db_dn_T = params_eq["db_dn_T"]

        bx, by, bz = t96_total_field(parmod_in, ps, X_eq_gsm, Y_eq_gsm, Z_eq_gsm)
        Bmag_nT = np.sqrt(bx**2 + by**2 + bz**2)

        FAC, ratio = compute_fac_proxy_and_ratio(Bmag_nT, dn_db_T, db_dn_T)

        fig = plt.figure(figsize=(16, 16))
        gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.25)
        axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]

        panels = [
            dict(
                data=dn_db_T,
                cmap="RdBu_r",
                pname="residual",
                formula="",
                title=r"(∂n/∂b)·T" + _gamma(r"$=\Gamma^{\hat{1}}_{\hat{3}\hat{2}}$"),
                cbar_label=r"(∂n/∂b)·T",
                unit="1/Re",
                force_levels=None,
            ),
            dict(
                data=-db_dn_T,
                cmap="RdBu_r",
                pname="residual",
                formula="",
                title=r"-(∂b/∂n)·T" + _gamma(r"$=-\Gamma^{\hat{1}}_{\hat{2}\hat{3}}$"),
                cbar_label=r"(∂b/∂n)·T",
                unit="1/Re",
                force_levels=None,
            ),
            dict(
                data=FAC,
                cmap="RdBu_r",
                pname="residual",
                formula="",
                title=(
                    r"Field-Aligned Current  $\frac{B}{\mu_0}$((∂n/∂b)·T − (∂b/∂n)·T)"
                ),
                cbar_label="FAC",
                unit="A/m$^2$",
                force_levels=None,
            ),
            dict(
                data=ratio,
                cmap="viridis",
                pname="ratio",
                formula=r"$\frac{|dn\_db\_T|}{|dn\_db\_T|+|db\_dn\_T|}$",
                title="Contribution ratio (0..1)",
                cbar_label="ratio",
                unit="",
                force_levels=np.linspace(0, 1, 21),
            ),
        ]

        for ax, P in zip(axes, panels):
            data = P["data"]
            if P["force_levels"] is not None:
                im = plot_scalar_on_grid(
                    ax,
                    X_sm,
                    Y_sm,
                    data,
                    P["cmap"],
                    P["pname"],
                    force_levels=P["force_levels"],
                )
            else:
                im = plot_scalar_on_grid(ax, X_sm, Y_sm, data, P["cmap"], P["pname"])

            if segments_eq is not None:
                add_micro_arrows(ax, segments_eq, alpha=0.8)
            if dot_x_eq.size > 0:
                ax.scatter(
                    dot_x_eq,
                    dot_y_eq,
                    s=dot_size,
                    marker="o",
                    c="k",
                    alpha=dot_alpha,
                    linewidths=0,
                )

            ax.add_patch(plt.Circle((0, 0), 1, color="white", zorder=10))
            ax.set_xlabel("X_SM (Re)")
            ax.set_ylabel("Y_SM (Re)")
            set_panel_title(ax, P["formula"], P["title"])
            ax.set_aspect("equal")
            ax.set_xlim(np.nanmin(X_sm), np.nanmax(X_sm))
            ax.set_ylim(np.nanmin(Y_sm), np.nanmax(Y_sm))

            add_colorbar(fig, ax, im, P["cbar_label"], unit=P["unit"])

            ax.text(
                0.02,
                0.98,
                f"[{np.nanmin(data):.2e}, {np.nanmax(data):.2e}]",
                transform=ax.transAxes,
                fontsize=9,
                va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
            )

        fig.tight_layout(h_pad=TIGHT_HPAD, w_pad=TIGHT_WPAD, pad=0.3)
        fig.subplots_adjust(top=0.92)
        fig.suptitle(
            "T96: Distribution of Field-Aligned Currents and \n Their Constituent Components in the Magnetic Equatorial Plane\n"
            f"{sw_title_line}",
            fontsize=16,
            y=0.985,
        )
        finish_figure(fig, f"FAC4_SM_z0_baseline__{sw_title_line}", show=True, close=True,
                      short_name=sn("FAC", "Equator_SM"))

    else:
        dn_db_T_new = _clip_small(derivatives_eq["dn_db_T"].reshape(X_sm.shape), delta_min)
        db_dn_T_new = _clip_small(derivatives_eq["db_dn_T"].reshape(X_sm.shape), delta_min)

        bx_new, by_new, bz_new = t96_total_field(parmod_in, ps, X_eq_gsm, Y_eq_gsm, Z_eq_gsm)
        Bmag_nT_new = np.sqrt(bx_new**2 + by_new**2 + bz_new**2)

        FAC_new, ratio_new = compute_fac_proxy_and_ratio(Bmag_nT_new, dn_db_T_new, db_dn_T_new)

        fig = plt.figure(figsize=(16, 15))
        gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.25)
        axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]

        panels_new = [
            dict(
                data=dn_db_T_new,
                cmap="RdBu_r",
                pname="residual",
                formula="",
                title=r"(∂n/∂b)·T" + _gamma(r"$=\Gamma^{\hat{1}}_{\hat{3}\hat{2}}$"),
                cbar_label=r"(∂n/∂b)·T",
                unit="1/Re",
                force_levels=None,
            ),
            dict(
                data=-db_dn_T_new,
                cmap="RdBu_r",
                pname="residual",
                formula="",
                title=r"-(∂b/∂n)·T" + _gamma(r"$=-\Gamma^{\hat{1}}_{\hat{2}\hat{3}}$"),
                cbar_label=r"(∂b/∂n)·T",
                unit="1/Re",
                force_levels=None,
            ),
            dict(
                data=FAC_new,
                cmap="RdBu_r",
                pname="residual",
                formula="",
                title=(
                    r"Field-Aligned Current  $\frac{B}{\mu_0}$((∂n/∂b)·T − (∂b/∂n)·T)"
                ),
                cbar_label="FAC",
                unit="A/m$^2$",
                force_levels=None,
            ),
            dict(
                data=ratio_new,
                cmap="viridis",
                pname="ratio",
                formula=r"$\frac{|dn\_db\_T|}{|dn\_db\_T|+|db\_dn\_T|}$",
                title="Contribution ratio (0..1)",
                cbar_label="ratio",
                unit="",
                force_levels=np.linspace(0, 1, 21),
            ),
        ]

        for ax, P in zip(axes, panels_new):
            data = P["data"]
            if P["force_levels"] is not None:
                im = plot_scalar_on_grid(
                    ax,
                    X_sm,
                    Y_sm,
                    data,
                    P["cmap"],
                    P["pname"],
                    force_levels=P["force_levels"],
                )
            else:
                im = plot_scalar_on_grid(ax, X_sm, Y_sm, data, P["cmap"], P["pname"])

            if segments_eq is not None:
                add_micro_arrows(ax, segments_eq, alpha=0.8)
            if dot_x_eq.size > 0:
                ax.scatter(
                    dot_x_eq,
                    dot_y_eq,
                    s=dot_size,
                    marker="o",
                    c="k",
                    alpha=dot_alpha,
                    linewidths=0,
                )

            ax.add_patch(plt.Circle((0, 0), 1, color="white", zorder=10))
            ax.set_xlabel("X_SM (Re)")
            ax.set_ylabel("Y_SM (Re)")
            set_panel_title(ax, P["formula"], P["title"])
            ax.set_aspect("equal")
            ax.set_xlim(np.nanmin(X_sm), np.nanmax(X_sm))
            ax.set_ylim(np.nanmin(Y_sm), np.nanmax(Y_sm))

            add_colorbar(fig, ax, im, P["cbar_label"], unit=P["unit"])

            ax.text(
                0.02,
                0.98,
                f"[{np.nanmin(data):.2e}, {np.nanmax(data):.2e}]",
                transform=ax.transAxes,
                fontsize=9,
                va="top",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
            )

        fig.tight_layout(h_pad=TIGHT_HPAD, w_pad=TIGHT_WPAD, pad=0.3)
        fig.subplots_adjust(top=0.92)
        fig.suptitle(
            "T96: FAC-related 4 panels in Magnetic Equatorial Plane (SM z=0, computed in GSM)\n"
            f"{sw_title_line}",
            fontsize=16,
            y=0.985,
        )
        finish_figure(fig, f"FAC4_SM_z0_{sw_title_line}", show=True, close=True,
                      short_name=sn("FAC", "Equator_SM"))

    # -------------------------
    # ADD: FAC-related 4 panels on GSM X-Z at Y_GSM = -3
    # -------------------------
    if case_mode == "baseline":
        plot_fac_4panels_xz_at_y(
            parmod_in,
            y0=-3.0,
            suptitle_prefix="T96: FAC-related 4 panels (baseline SW params)",
            sw_title_line=sw_title_line,
            short_name=sn("FAC", "Yminus3_GSM"),
        )
    else:
        plot_fac_4panels_xz_at_y(
            parmod_in,
            y0=-3.0,
            suptitle_prefix="T96: FAC-related 4 panels",
            sw_title_line=sw_title_line,
            short_name=sn("FAC", "Yminus3_GSM"),
        )

    # -------------------------
    # ADD: FAC-related 4 panels on SM z=0.5 plane
    # -------------------------
    plot_fac_4panels_sm_z_plane(
        parmod_in,
        z_sm0=0.5,
        sw_title_line=sw_title_line,
        suptitle_prefix="T96: FAC-related 4 panels in Magnetic Equatorial Parallel Plane (SM z=0.5)",
        short_name=sn("FAC", "SMz05_SM"),
    )


---

MAIN: Run both Baseline and NEW for BY=0nT and BY=5nT (after changes)

---

In [ ]:
# -------------------------
# Baseline SW params: Run both By = 0 nT and 5 nT
# -------------------------
BASE_SW = dict(density_cm3=10.0, v_kms=600.0, dst_nT=-30.0, bz_nT=-5.0)

for ByIMF in BY_VALUES:
    parmod, Pdyn = make_parmod_from_sw(
        density_cm3=BASE_SW["density_cm3"],
        v_kms=BASE_SW["v_kms"],
        dst_nT=BASE_SW["dst_nT"],
        by_nT=ByIMF,
        bz_nT=BASE_SW["bz_nT"],
    )

    sw_title = (
        f"Pdyn={parmod[0]:.2f} nPa, "
        f"Dst={parmod[1]:.0f} nT, "
        f"ByIMF={parmod[2]:.1f} nT, "
        f"BzIMF={parmod[3]:.1f} nT"
    )

    print("\nT96 parameters:")
    print(f"  Pdyn  = {parmod[0]:.2f} nPa")
    print(f"  Dst   = {parmod[1]:.1f} nT")
    print(f"  ByIMF = {parmod[2]:.1f} nT")
    print(f"  BzIMF = {parmod[3]:.1f} nT")

    run_all_figures_for_case(parmod, sw_title, case_mode="baseline")


# -------------------------
# NEW SW params: Run both By = 0 nT and 5 nT
# -------------------------
NEW_SW = dict(density_cm3=15.0, v_kms=650.0, dst_nT=-150.0, bz_nT=-15.0)

for ByIMF_new in BY_VALUES:
    parmod_new, Pdyn_new = make_parmod_from_sw(
        density_cm3=NEW_SW["density_cm3"],
        v_kms=NEW_SW["v_kms"],
        dst_nT=NEW_SW["dst_nT"],
        by_nT=ByIMF_new,
        bz_nT=NEW_SW["bz_nT"],
    )

    sw_title_new = (
        f"Pdyn={Pdyn_new:.2f} nPa, Dst={NEW_SW['dst_nT']:.0f} nT, "
        f"ByIMF={ByIMF_new:.1f} nT, BzIMF={NEW_SW['bz_nT']:.1f} nT"
    )
    print("\n" + sw_title_new)

    run_all_figures_for_case(parmod_new, sw_title_new, case_mode="new")
